In [166]:
Features = [
    'item_nbr',
    'family',
    'class',
    'perishable',
    'has_float_sales',
    'is_likely_closed',
    'day_of_week',
    'is_weekend',
    'is_month_start',
    'is_month_end',
    'item_mean_sales',
    'item_std_sales',
    'sales_vs_family_mean',
    'lag_1',
    'lag_3',
    'lag_5',
    'lag_7',
    'lag_14',
    'lag_30',
    'lag_60',
    'lag_140',
    'rolling_mean_3',
    'rolling_mean_7',
    'rolling_mean_14',
    'rolling_mean_30',
    'rolling_mean_140',
    'rolling_std_7',
    'rolling_std_14',
    'rolling_std_30',
    'rolling_max_7',
    'rolling_min_7',
    'transactions_lag_1',
    'transactions_lag_3',
    'transactions_lag_7',
    'transactions_lag_14',
    'transactions_rolling_7',
    'transactions_rolling_14',
    'transactions_rolling_30',
    'traffic_ratio',
    'oil_price_lag_1',
    'oil_price_lag_3',
    'oil_price_lag_7',
    'oil_price_rolling_mean_7',
    'oil_price_rolling_mean_30',
    'oil_price_change_7',
    'promo_lag_1',
    'promo_lag_3',
    'promo_lag_7',
    'promo_count_7',
    'promo_count_30',
    'days_since_last_promo',
    'days_since_last_holiday',
    'days_to_next_holiday',
    'Future_is_Holiday',
    'Future_is_National',
    'Future_is_Local',
    'Future_is_likely_closed',
    'Future_day_of_week',
    'Future_is_weekend',
    'Future_is_month_start',
    'Future_is_month_end',
    'Future_On_Promotion'
]
len(Features)

62

In [167]:
Target = 'target'

In [168]:
Categorical_Features = [
    'item_nbr',
    'family',
    'class'
]

In [169]:
Future_Features_Map = {
    'is_Holiday' : 'Future_is_Holiday',
    'is_National' : 'Future_is_National',
    'is_Local' : 'Future_is_Local',
    'is_likely_closed' : 'Future_is_likely_closed',
    'day_of_week' : 'Future_day_of_week',
    'is_weekend' : 'Future_is_weekend',
    'is_month_start' : 'Future_is_month_start',
    'is_month_end' : 'Future_is_month_end',
    'onpromotion' : 'Future_On_Promotion',
    'unit_sales' : 'target'
}
len(Future_Features_Map)

10

In [170]:
horizon = 28

In [171]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

transformer = FunctionTransformer(func=np.log1p, inverse_func=np.expm1)

In [172]:
from xgboost import XGBRegressor
import polars as pl
from tqdm import trange

In [173]:
def NWRMSLE_Evaluator(Dataset : pl.DataFrame, horizon : int, horizon_total : int, transformer : FunctionTransformer):
    mapping = {
        0 : 1.0,
        1 : 1.5
    }
    Added_Future_Features_Dataset = Dataset.with_columns(
            [
                pl.col(oldName).shift(-(horizon)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
            ]
        ).with_columns(
            [
                pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
            ]
        ).with_columns(
            [
                pl.col('perishable').replace_strict(mapping, default=None).alias('weight'),
                pl.col('item_nbr').shift(-horizon_total).over('item_nbr').alias('delete')
            ]
        )

    Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

    rounding_mask = Added_Future_Features_Dataset.get_column('has_float_sales').to_numpy()

    X = Added_Future_Features_Dataset.select(Features).to_pandas()
    y = Added_Future_Features_Dataset.get_column(Target).to_numpy()
    weights = Added_Future_Features_Dataset.get_column('weight').to_numpy()
    
    def NWRMSLE(y_true, y_pred):
        y_pred = transformer.inverse_transform(y_pred)
        y_pred = np.clip(y_pred, a_min=0, a_max=None)
        y_pred = np.where(rounding_mask == 0, np.round(y_pred), y_pred)
        y_pred_transformed = transformer.transform(y_pred)

        loss = (weights*(y_pred_transformed - y_true)**2).sum()/weights.sum()

        return loss
    
    return NWRMSLE, X, y

In [174]:
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

In [176]:
from optuna.integration.xgboost import XGBoostPruningCallback

def train_trial(trial, Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator):

    pruning_callback = XGBoostPruningCallback(trial, observation_key="validation_0-NWRMSLE")

    parameters = {
        # Tree structure
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_weight': trial.suggest_float('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0.0, 10.0),

        # Learning
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 300, 1000),

        # Sampling
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 50.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 50.0, log=True),

        # GPU-specific
        'max_bin': trial.suggest_int('max_bin', 256, 1024),

        'objective': 'reg:squarederror',
        'tree_method': 'hist',
        'device': 'cuda',
        'eval_metric': nwrmsle_Evaluator,
        'enable_categorical': True,
        'n_jobs': -1,
        'verbosity': 0,
        'callbacks': [pruning_callback]
    }

    model = XGBRegressor(**parameters)

    model.fit(
        Train_X,
        Train_Y,
        eval_set=[(Val_X, Val_Y)],
        # eval_metric=nwrmsle_Evaluator,
        verbose=False
    )

    score = model.evals_result_["validation_0"]["NWRMSLE"][-1]

    return score

def make_objective(Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator):
    def objective(trial):
        return train_trial(trial, Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator)
    return objective


In [177]:
Train_Dataset = pl.read_csv('Store39_Dataset/MasterTrain.csv')
Val_Dataset = pl.read_csv("Store39_Dataset/MasterVal.csv")

In [178]:
import os
optuna_path = f"./optuna_studies_XGB"
os.makedirs(optuna_path, exist_ok=True)

In [ ]:
import joblib

N_TRIALS = 15

for h in trange(horizon, desc='Currently Tuning Model'):
    Added_Future_Features_Dataset = Train_Dataset.with_columns(
            [
                pl.col(oldName).shift(-(h+1)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
            ]
        ).with_columns(
            [
                pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
            ]
        ).with_columns(
            pl.col('item_nbr').shift(-horizon).over('item_nbr').alias('delete')
        )

    Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

    Train_X = Added_Future_Features_Dataset.select(Features).to_pandas()
    Train_Y = Added_Future_Features_Dataset.get_column(Target).to_numpy()

    nwrmsle_Evaluator, Val_X, Val_Y = NWRMSLE_Evaluator(Val_Dataset, h, horizon, transformer)

    sampler = TPESampler(seed=42, multivariate=True)

    pruner = MedianPruner(
        n_startup_trials = 5,
        n_warmup_steps = 30
    )

    study = optuna.create_study(
        direction = 'minimize',
        sampler = sampler,
        pruner = pruner,
        study_name= f'Model{h}_tuning'
    )
 
    study.optimize(
        make_objective(Train_X, Train_Y, Val_X, Val_Y, nwrmsle_Evaluator),
        n_trials  = N_TRIALS,
        show_progress_bar = True
    )

    print(f"\n ============== Model{h} Results ==============")
    print("Best trial:")
    print(f"Val loss : {study.best_trial.value:.6f}")
    print(f"Params :")
    for k, v in study.best_trial.params.items():
        print(f"{k:20s} : {v}")

    joblib.dump(study, optuna_path+f'/Optuna_Model{h+1}_Study.pkl')

In [193]:
for i in range(1, horizon + 1):
    print(f'=========== Model {i} Hyper Parameters ==============')
    study = joblib.load(f"./optuna_studies_xgb/Optuna_Model{i}_Study.pkl")
    for k, v in study.best_trial.params.items():
        print(f"{k:20s} : {v}")

=========== Model 1 Hyper Parameters ==============
max_depth            : 6
min_child_weight     : 19.26371376651486
gamma                : 7.448557290856435
learning_rate        : 0.05040958967382934
n_estimators         : 387
subsample            : 0.6413768898285196
colsample_bytree     : 0.6402274806521797
colsample_bylevel    : 0.9756656414339995
reg_alpha            : 5.2334841377372285
reg_lambda           : 0.0006594630768039654
max_bin              : 490
=========== Model 2 Hyper Parameters ==============
max_depth            : 7
min_child_weight     : 19.063571821788408
gamma                : 7.319939418114051
learning_rate        : 0.045504758132021865
n_estimators         : 409
subsample            : 0.662397808134481
colsample_bytree     : 0.6232334448672797
colsample_bylevel    : 0.9464704583099741
reg_alpha            : 0.006763888939818983
reg_lambda           : 0.07371819500318835
max_bin              : 271
=========== Model 3 Hyper Parameters ==============
max_depth